# just_tx vs just_tx_hal：处理 PPS

同一块 ESP32-C3、同一 AP、同一台 TR3000 上比较两条发送路径。HAL 是 `just_tx_hal`（应用层 UDP:8080 原载荷回发，成功则计 `rxcnt[17]`）；XDP 是 `just_drop` 固件加冷表 `just_tx.elf`（Wi-Fi 入口 MAC 交换 + `XDP_TX`，程序里给 `rxcnt[17]` 加一）。计的是进入各自发送路径的包。

三个指标与 `just_drop_pps` 相同：平均处理 PPS、1 s 相对 ADEV、跨轮 CV。本笔记只读 `source_note = xpost-test just-tx-pps` 的 `formal.experiment_*` 行。

复现：Mac 上 `cargo xtask agent --listen 0.0.0.0:9240 --no-monitor`（cwd = `xpost/`）；TR3000 上 `./xpost-test just-tx-pps`。


In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import (
    cv_bootstrap_ci,
    figure1_summary,
    load_counter_samples,
    load_runs,
    mean_across_runs,
    mean_pps,
    mean_t_ci,
    overlapping_relative_adev,
    summarize_variant,
)

SOURCE_NOTE = "xpost-test just-tx-pps"

plt.ioff()
plt.close("all")
print(dsn_summary())

runs = load_runs(run_kind="pilot", source_note=SOURCE_NOTE)
assert len(runs), (
    "formal.experiment_runs 无 just-tx-pps pilot 行：先在 TR3000 跑 "
    "./xpost-test just-tx-pps，且 .env 设 PGSCHEMA=formal"
)
samples = load_counter_samples(runs["run_id"].tolist())
print(f"runs={len(runs)} samples={len(samples)}")

by_variant = defaultdict(list)
adev_runs = defaultdict(list)

for _, row in runs.iterrows():
    sub = samples[samples["run_id"] == row["run_id"]].sort_values("time_ns")
    t_ns = sub["time_ns"].tolist()
    c = sub["counter_value"].tolist()
    variant = row["variant"]
    by_variant[variant].append(
        mean_pps(t_ns, c, run_id=row["run_id"], variant=variant, pair_id=int(row["pair_id"]))
    )
    adev = overlapping_relative_adev(t_ns, c)
    if adev:
        adev_runs[variant].append({tau: d["adev"] for tau, d in adev.items()})

summaries = {v: summarize_variant(rs) for v, rs in by_variant.items()}
adev_pool = {v: mean_across_runs(rows) for v, rows in adev_runs.items()}


def _at(pool, tau=1.0):
    row = pool.get(tau) or {}
    return row.get("mean"), (row.get("lo"), row.get("hi"))


for v, s in summaries.items():
    a_mean, (a_lo, a_hi) = _at(adev_pool.get(v, {}))
    print(
        f"{v}: valid={s['n_valid']}/{s['n_total']}  "
        f"PPS={s['mean_pps']:.0f}  ADEV@1s={100 * a_mean:.2f}% "
        f"({100 * a_lo:.2f}–{100 * a_hi:.2f})  "
        f"CV={100 * s['cv']:.2f}%  "
        f"per-run={[round(x) for x in s['per_run_pps']]}"
    )

hal, xdp = summaries.get("hal", {}), summaries.get("xdp", {})
hal_pps = hal.get("per_run_pps") or []
xdp_pps = xdp.get("per_run_pps") or []
_, hal_pps_lo, hal_pps_hi = mean_t_ci(hal_pps)
_, xdp_pps_lo, xdp_pps_hi = mean_t_ci(xdp_pps)
hal_cv, _, _ = cv_bootstrap_ci(hal_pps)
xdp_cv, _, _ = cv_bootstrap_ci(xdp_pps)
hal_a, hal_a_ci = _at(adev_pool.get("hal", {}))
xdp_a, xdp_a_ci = _at(adev_pool.get("xdp", {}))

fig, _ = figure1_summary(
    hal_mean_pps=hal.get("mean_pps", float("nan")),
    xdp_mean_pps=xdp.get("mean_pps", float("nan")),
    hal_pps_ci=(hal_pps_lo, hal_pps_hi),
    xdp_pps_ci=(xdp_pps_lo, xdp_pps_hi),
    hal_adev_1s=hal_a,
    xdp_adev_1s=xdp_a,
    hal_adev_ci=hal_a_ci,
    xdp_adev_ci=xdp_a_ci,
    hal_cv=hal_cv,
    xdp_cv=xdp_cv,
    title="just_tx HAL vs XDP",
)
display(fig)
plt.close(fig)
